# Boston Housing — Entrenamiento y Selección de Modelos

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

project_root = Path.cwd().parent
load_dotenv(project_root / ".env")

SQLITE_PATH = project_root / os.environ.get("SQLITE_PATH", ".storage/data.db")
TABLE_NAME = "housing_data"
MLFLOW_TRACKING_URI = "http://localhost:5001"

EXPERIMENT_NAME = "boston-housing-experiments"
MODEL_REGISTRY_NAME = "boston-housing-regressor"

STAGING_ALIAS = "staging"
PRODUCTION_ALIAS = "production"

os.environ["AWS_ACCESS_KEY_ID"] = os.environ.get("MINIO_ROOT_USER", "")
os.environ["AWS_SECRET_ACCESS_KEY"] = os.environ.get("MINIO_ROOT_PASSWORD", "")
os.environ["MLFLOW_S3_ENDPOINT_URL"] = os.environ.get(
    "MINIO_ENDPOINT", "http://localhost:9000"
)


print(f"MLflow URI: {MLFLOW_TRACKING_URI}")
print(f"MinIO endpoint: {os.environ['MLFLOW_S3_ENDPOINT_URL']}")

MLflow URI: http://localhost:5001
MinIO endpoint: http://localhost:9000


In [2]:
# Dependencias necesarias
import numpy as np
import pandas as pd

import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from src.data.repository import load_data
from src.data.schema import FEATURE_COLUMNS, TARGET_COLUMN

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

/Users/dalzoj/Documents/IA/boston-housing-mlops/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Carga inicial de datos
df = load_data(SQLITE_PATH, "housing_data")
print("Tamaño:", df.shape)

# Eliminar vacíos
df.dropna(axis=0, inplace=True)

X = df[FEATURE_COLUMNS]
y = df[TARGET_COLUMN]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.1, random_state=42
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

Tamaño: (1006, 15)
Train: (698, 13), Test: (78, 13)


In [4]:
# Evaluar modelo
def evaluate(pipeline, X_test, y_test):
    y_pred = pipeline.predict(X_test)
    return {
        "rmse": float(np.sqrt(mean_squared_error(y_test, y_pred))),
        "mae": float(mean_absolute_error(y_test, y_pred)),
        "r2": float(r2_score(y_test, y_pred)),
    }


# Construir preprocesador
def build_preprocessor():
    numeric_pipeline = Pipeline(
        steps=[
            ("scaler", StandardScaler()),
        ]
    )
    return ColumnTransformer(
        transformers=[("num", numeric_pipeline, FEATURE_COLUMNS)],
        remainder="drop",
        verbose_feature_names_out=False,
    )

In [5]:
# Experimento en MlFlow
mlflow.set_experiment(EXPERIMENT_NAME)

print(f"Experiment: {EXPERIMENT_NAME}")

2026/05/14 18:29:11 INFO mlflow.tracking.fluent: Experiment with name 'boston-housing-experiments' does not exist. Creating a new experiment.


Experiment: boston-housing-experiments


In [6]:
# Entrenamiento con Ridge
ridge_params = {"alpha": 1.0, "random_state": 42}

ridge_pipeline = Pipeline(
    steps=[
        ("preprocessor", build_preprocessor()),
        ("estimator", Ridge(**ridge_params)),
    ]
)

with mlflow.start_run(run_name="ridge") as run:
    mlflow.set_tag("model_name", "ridge")
    mlflow.log_params(ridge_params)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("test_size", 0.2)

    ridge_pipeline.fit(X_train, y_train)
    ridge_metrics = evaluate(ridge_pipeline, X_test, y_test)
    mlflow.log_metrics(ridge_metrics)

    mlflow.sklearn.log_model(
        sk_model=ridge_pipeline,
        artifact_path="model",
        input_example=X_train.iloc[:3],
    )

    ridge_run_id = run.info.run_id

print(f"Ridge run_id: {ridge_run_id}")
print(f"Ridge métricas: {ridge_metrics}")

2026/05/14 18:29:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/14 18:29:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
/Users/dalzoj/Documents/IA/boston-housing-mlops/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that inclu

🏃 View run ridge at: http://localhost:5001/#/experiments/2/runs/5d5bbcf8573042be82aee040b1badd92
🧪 View experiment at: http://localhost:5001/#/experiments/2
Ridge run_id: 5d5bbcf8573042be82aee040b1badd92
Ridge métricas: {'rmse': 4.706917924207046, 'mae': 3.461529122127866, 'r2': 0.6956939108538414}


In [7]:
# Entrenamiento con Descenso del Gradiente

gb_params = {
    "n_estimators": 200,
    "max_depth": 4,
    "learning_rate": 0.05,
    "random_state": 42,
}

gb_pipeline = Pipeline(
    steps=[
        ("preprocessor", build_preprocessor()),
        ("estimator", GradientBoostingRegressor(**gb_params)),
    ]
)

with mlflow.start_run(run_name="gradient_boosting") as run:
    mlflow.set_tag("model_name", "gradient_boosting")
    mlflow.log_params(gb_params)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("test_size", 0.2)

    gb_pipeline.fit(X_train, y_train)
    gb_metrics = evaluate(gb_pipeline, X_test, y_test)
    mlflow.log_metrics(gb_metrics)

    mlflow.sklearn.log_model(
        sk_model=gb_pipeline,
        artifact_path="model",
        input_example=X_train.iloc[:3],
    )

    gb_run_id = run.info.run_id

print(f"GBM run_id: {gb_run_id}")
print(f"GBM mpetricas: {gb_metrics}")

2026/05/14 18:29:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/14 18:29:16 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
/Users/dalzoj/Documents/IA/boston-housing-mlops/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that inclu

🏃 View run gradient_boosting at: http://localhost:5001/#/experiments/2/runs/2d2c47e5aba244af8ed57afea4c9556a
🧪 View experiment at: http://localhost:5001/#/experiments/2
GBM run_id: 2d2c47e5aba244af8ed57afea4c9556a
GBM mpetricas: {'rmse': 3.94722783420058, 'mae': 3.012206289473997, 'r2': 0.7859960541856319}


In [8]:
# Comparar mejor modelo
results = [
    {"model": "ridge", "run_id": ridge_run_id, **ridge_metrics},
    {"model": "gradient_boosting", "run_id": gb_run_id, **gb_metrics},
]

results_df = pd.DataFrame(results).set_index("model")
print(results_df.round(4))

winner = min(results, key=lambda r: r["rmse"])
print(f"Mejor modelo: {winner['model']} con RMSE={winner['rmse']:.4f}")

                                             run_id    rmse     mae      r2
model                                                                      
ridge              5d5bbcf8573042be82aee040b1badd92  4.7069  3.4615  0.6957
gradient_boosting  2d2c47e5aba244af8ed57afea4c9556a  3.9472  3.0122  0.7860
Mejor modelo: gradient_boosting con RMSE=3.9472


In [9]:
# Registrar modelo y promover a Staging
model_uri = f"runs:/{winner['run_id']}/model"
version = mlflow.register_model(model_uri=model_uri, name=MODEL_REGISTRY_NAME)
print(f"Versión registrada: {version.version}")

client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)
client.set_registered_model_alias(
    name=MODEL_REGISTRY_NAME,
    alias=STAGING_ALIAS,
    version=version.version,
)

print(f"'{MODEL_REGISTRY_NAME}' v{version.version} ahora fue promovido a STAGING")

Registered model 'boston-housing-regressor' already exists. Creating a new version of this model...
2026/05/14 18:29:18 WARNING mlflow.tracking._model_registry.fluent: Run with id 2d2c47e5aba244af8ed57afea4c9556a has no artifacts at artifact path 'model', registering model based on models:/m-806ecee937814e4c81b80c8a0a581cdf instead
2026/05/14 18:29:18 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: boston-housing-regressor, version 7


Versión registrada: 7
'boston-housing-regressor' v7 ahora fue promovido a STAGING


Created version '7' of model 'boston-housing-regressor'.
